In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "knoveleng/Open-RS2"

print("Đang tải tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tải bản sao thứ nhất lên GPU 0
print("Đang tải model bản sao 1 lên GPU 0 (cuda:0)...")
model_0 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16, # Bắt buộc dùng float16 cho card T4
    device_map={'': 'cuda:0'}
)

# Tải bản sao thứ hai lên GPU 1
print("Đang tải model bản sao 2 lên GPU 1 (cuda:1)...")
model_1 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map={'': 'cuda:1'}
)

print("Tải model hoàn tất! Sẵn sàng cho 2 GPU.")

Đang tải tokenizer...


config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/485 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Đang tải model bản sao 2 lên GPU 1 (cuda:1)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Tải model hoàn tất! Sẵn sàng cho 2 GPU.


In [ ]:
import pandas as pd

# Đường dẫn tới file jsonl của bạn
file_path = "/kaggle/input/datasets/frankquoc/svamp/SVAMP_eval_ready.jsonl"
df = pd.read_json(file_path, lines=True)


# Đảm bảo số lượng câu hỏi luôn là số chẵn để chia đều cho 2 GPU
if len(df) % 2 != 0:
    df = df.iloc[:-1]

questions = df['Question'].tolist()
ground_truths = df['Answer'].tolist()

print(f"Đã tải xong dữ liệu")

Đã tải xong dữ liệu


In [ ]:
import re
import concurrent.futures
from tqdm.notebook import tqdm

# Ép model luôn bỏ kết quả cuối cùng vào \boxed{} để dùng Regex tìm
system_prompt = r"""You are a strict mathematical calculator. Solve the problem using pure algebraic logic based EXACTLY on the text provided.
Strict Rules:
1. IGNORE real-world physical constraints. It is strictly ALLOWED to have negative distances, fractional items, or physically illogical scenarios.
2. DO NOT correct or reinterpret the problem if the numbers seem illogical. Calculate exactly what is given.
3. Keep your reasoning brief and step-by-step.
4. Put your final answer within \boxed{}.
"""

# Hàm chạy suy luận trên một GPU cụ thể
def process_on_gpu(question, model, device):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=4096,
            temperature=0.3,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )
    # Lấy phần text mới được sinh ra
    response_ids = outputs[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(response_ids, skip_special_tokens=True)

# Hàm dùng Regex để tìm đáp án trong \boxed{}
def extract_answer(text):
    match = re.search(r"\\boxed\{([^}]*)\}", text)
    if match:
        return match.group(1).strip()
    return None

correct_count = 0
total_count = len(questions)
failed_example = None


# Vòng lặp nhảy bước 2 (để lấy 2 câu mỗi lần)
for i in tqdm(range(0, total_count, 2)):
    q1 = questions[i]
    q2 = questions[i+1]
    
    # Mở 2 luồng (thread) để bắt 2 GPU làm việc cùng một lúc
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        future_0 = executor.submit(process_on_gpu, q1, model_0, "cuda:0")
        future_1 = executor.submit(process_on_gpu, q2, model_1, "cuda:1")
        
        # Chờ cả 2 GPU trả về kết quả
        ans_text_1 = future_0.result()
        ans_text_2 = future_1.result()

    # Đánh giá câu 1
    extracted_1 = extract_answer(ans_text_1)
    truth_1 = str(ground_truths[i]).strip()
    if extracted_1 == truth_1:
        correct_count += 1
    elif failed_example is None: # Lưu lại câu sai đầu tiên
        failed_example = (q1, truth_1, extracted_1, ans_text_1)

    # Đánh giá câu 2
    extracted_2 = extract_answer(ans_text_2)
    truth_2 = str(ground_truths[i+1]).strip()
    if extracted_2 == truth_2:
        correct_count += 1
    elif failed_example is None:
        failed_example = (q2, truth_2, extracted_2, ans_text_2)

# ==========================================
# IN KẾT QUẢ ĐÚNG YÊU CẦU
# ==========================================
success_rate = (correct_count / total_count) * 100
print("\n" + "="*50)
print(f"Tỉ lệ thành công: {success_rate:.2f}%")
print("="*50)

if failed_example:
    q, truth, ext, full_ans = failed_example
    print("\n=== MỘT CÂU TRẢ LỜI SAI MẪU ===")
    print(f"[Câu hỏi]:\n{q}")
    print("-" * 30)
    print(f"[Kết quả thực sự]: {truth}")
    print(f"[Kết quả Model đã đưa ra (trích xuất được)]: {ext}")
    print("-" * 30)
    print("[Toàn bộ lập luận của Model]:\n")
    print(full_ans)

Could not render content for 'application/vnd.jupyter.widget-view+json'
{"model_id":"e08ef7fac7f34192832a78e4cad3a88b","version_major":2,"version_minor":0}
Tỉ lệ thành công: 80.57%

=== MỘT CÂU TRẢ LỜI SAI MẪU ===
[Câu hỏi]:
Together Adam and Jackie have 12 apples. He has 9 apples more than Adam and Jackie together do. Adam has 8 more apples than Jackie. How many apples does He have?
------------------------------
[Kết quả thực sự]: 21
[Kết quả Model đã đưa ra (trích xuất được)]: -9
------------------------------
[Toàn bộ lập luận của Model]:

First, I need to determine the total number of apples that Adam and Jackie have together. According to the problem, together they have 12 apples.

Next, it's stated that Jack has 9 apples more than the combined total of Adam and Jackie. This means Jack has 12 + 9 = 21 apples.

Let A represent the number of apples Adam has and J represent the number of apples Jack has. We know that A + J = 12 and J = 21.

Substituting J = 21 into the first equatio